# Week 11 Lecture: Tree-Based Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/notebooks/tuesday-your-turn/week-11-lecture.ipynb)

This week you'll build a decision tree, combine many trees into a random forest, and use feature importance and partial dependence plots to explain what a forest has learned.

## How to use this notebook

This notebook accompanies the Week 11 Tuesday lecture. You can:

- **Follow along during class**: run each cell as the instructor discusses it
- **Work the Your Turn exercises** when prompted

## Setup

Today's data comes from the `ISLP` package, which isn't installed in Colab by default. Run this cell first.

> ⏳ The install takes about a minute. If Colab asks you to **restart the session**, click **Restart**, then continue with the next cell (you don't need to rerun this one).

In [ ]:
%pip install ISLP --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ISLP import load_data
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, recall_score, root_mean_squared_error
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

---

## 1. Build a Decision Tree

Same credit card `Default` data and split as last week. Trees need numeric inputs, so convert the Yes/No columns to 1/0.

In [ ]:
Default = load_data("Default")
Default["default"] = (Default["default"] == "Yes").astype(int)
Default["student"] = (Default["student"] == "Yes").astype(int)

X = Default[["student", "balance", "income"]]
y = Default["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

### Reading the tree

Each box shows the **question** (True → left, False → right), its **gini** (0 = pure), the training **samples** that reach it, the **value** counts [paid, defaulted], and the **class** it predicts.

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(tree_model, feature_names=list(X.columns), class_names=["No Default", "Default"],
          filled=True, rounded=True, precision=2, fontsize=9)
plt.show()

### Making a prediction

A new customer drops in at the top and follows the answers down to a leaf:

In [ ]:
customer = pd.DataFrame({
    "student": [0],
    "balance": [2000],
    "income": [40000]
})

print("Predicted class:", tree_model.predict(customer))
print("P(no default), P(default):", tree_model.predict_proba(customer).round(2))

### Did the tree beat the lazy model?

Same check as last week: compare to predicting "No default" for everyone.

In [ ]:
y_pred = tree_model.predict(X_test)
lazy_pred = np.zeros(len(y_test))   # predict "No default" for everyone

print(f"Lazy model accuracy: {accuracy_score(y_test, lazy_pred):.1%}")
print(f"Tree accuracy:       {accuracy_score(y_test, y_pred):.1%}")
print(f"Tree recall:         {recall_score(y_test, y_pred):.2f}")

### 🎯 Your Turn: How Deep Should the Tree Go?

1. Fit trees with `max_depth` of **2, 5, 10, and `None`** (no limit). For each, print the **training accuracy**, **test accuracy**, and **test recall**.
2. What happens to training accuracy as the tree gets deeper? What happens to test accuracy?

*Hint: loop over `[2, 5, 10, None]` and reuse the code above. `model.score(X, y)` returns accuracy.*

In [ ]:
# Your code here

*Your answer to question 2:*

---

## 2. Single Tree vs. Random Forest

Now predict a **number**: median home value (`medv`, in $1,000s) in the Boston housing data. Both models are kept shallow (`max_depth=3`) for illustration. `n_estimators` sets how many trees are in the forest.

In [ ]:
Boston = load_data("Boston")
X = Boston.drop(columns="medv")
y = Boston["medv"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

single_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
single_tree.fit(X_train, y_train)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=3, random_state=42)
rf_model.fit(X_train, y_train)

Evaluate them the same way as week 9: train vs. test R² and test RMSE.

In [ ]:
results = []
for name, model in [("Single tree", single_tree), ("Random forest", rf_model)]:
    results.append({
        "model": name,
        "train R²": model.score(X_train, y_train),
        "test R²": model.score(X_test, y_test),
        "test RMSE ($K)": root_mean_squared_error(y_test, model.predict(X_test))
    })

pd.DataFrame(results).round(3)

---

## 3. Which Features Matter, and How?

### Gini importance

Built into every tree-based model: each feature's **share** of the total impurity drop across all the trees (computed on the training data).

In [ ]:
gini_importance = (
    pd.DataFrame({"feature": X_train.columns, "importance": rf_model.feature_importances_})
    .sort_values("importance", ascending=False)
)

gini_importance.head()

### Permutation importance

Shuffle one column of the **test** data at a time and measure the average drop in test R².

In [ ]:
perm_result = permutation_importance(
    rf_model, X_test, y_test, n_repeats=10, random_state=42, scoring="r2"
)
perm_importance = (
    pd.DataFrame({"feature": X_test.columns, "importance": perm_result.importances_mean})
    .sort_values("importance", ascending=False)
)
perm_importance.head()

Both methods put `rm` (average rooms) and `lstat` (% lower-status residents) on top, but in a different order.

### Partial dependence plots

Set every home's `rm` to the same value, keep everything else, and average the forest's predictions. Thin lines are individual homes; the dashed line is their average (the PDP).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
pdp = PartialDependenceDisplay.from_estimator(
    rf_model, X_train, features=["rm"], kind="both",
    subsample=50, random_state=42, ax=ax
)
pdp.axes_[0, 0].set_ylabel("Predicted value ($1,000s)")
plt.show()

Plot several features side by side to compare their shapes:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
PartialDependenceDisplay.from_estimator(
    rf_model, X_train, features=["rm", "lstat", "dis", "crim"], ax=axes
)
plt.tight_layout()
plt.show()

### 🎯 Your Turn: Explain It to the VP

1. Create a PDP (with `kind="both"`) for `lstat`.
2. In one or two sentences a non-technical manager would understand, describe how `lstat` affects the predicted home value. Is there a point where the effect levels off?

In [ ]:
# Your code here

*Your answer to question 2:*

---

## 🧾 What You Learned

| Code | What it does |
|------|--------------|
| `DecisionTreeClassifier(max_depth=3)` | A classification tree limited to 3 levels of questions |
| `DecisionTreeRegressor()` | A tree that predicts a number (each leaf predicts an average) |
| `plot_tree(model, feature_names=..., filled=True)` | Draw a fitted tree |
| `model.predict(X)`, `model.predict_proba(X)` | Predicted class, and class probabilities from the leaf |
| `RandomForestRegressor(n_estimators=100)` | A forest of 100 trees whose predictions are averaged |
| `model.feature_importances_` | Gini (impurity-based) importance from training |
| `permutation_importance(model, X_test, y_test)` | Drop in test performance when each feature is shuffled |
| `PartialDependenceDisplay.from_estimator(model, X, features=[...])` | How predictions change as one feature changes |